In [1]:
from NN_functions import *
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import tensorflow as tf 

In [2]:
?simple_NN

Signature: simple_NN(X, y, seed=2, lr=0.1, hidden_size=5, max_iter=30, **kwargs)
Docstring:
X, y should be numpy arrays
W1_init: optional argument for initialized weight
W2_init: optional argument for initialized weight
File:      ~/Documents/GIT/MTDS/notebooks/NN_functions.py
Type:      function

## A comment on `**kwargs`
`**kwargs` stands for "keyword arguments". It should be placed at the end of arguments and provides optional arguments in the format of dictionary.

Below is a quick example

In [91]:
def test_kwargs(**kwargs):
    name_default = "John"

    # if there is no keyword "name" in kwargs, then use name_default
    name = kwargs.get('name', name_default)
    return name

In [92]:
test_kwargs()

'John'

In [94]:
test_kwargs(name = 'Mike')

'Mike'

In [20]:
# data prep
ames = pd.read_csv("../data/ames.csv")
ames_num = ames.select_dtypes(exclude=['object'])
yh = ames['Sale_Price']
Xh = ames_num.iloc[:,:-3] # selecting all but the last 3 columns
X_train, X_test, y_train, y_test = train_test_split(Xh, yh, test_size = 0.25, random_state = 45)
X_train_n = X_train.to_numpy()
y_train_n = y_train.to_numpy()
X_test_n = X_test.to_numpy()
y_test_n = y_test.to_numpy()

## Feed `simple_NN()` with trained weights

below, we have split these 30K iterations into 3 portions. We feed the previously trained weights into the next portion.

In [21]:
p1 = 5

In [22]:
# 3e4
W1_1, W2_1, J_1, u2_1 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4))
print(J_1**0.5)
print(root_mean_squared_error(u2_1, y_train_n))

40472.64581352626
40472.64581352626


In [23]:
W1_2, W2_2, J_2, u2_2 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4),
                                 W1_init = W1_1,
                                 W2_init = W2_1)
print(J_2**0.5)

40076.11782201403


In [24]:
W1_3, W2_3, J_3, u2_3 = simple_NN(X_train_n, y_train_n, hidden_size = p1, seed = 3, lr = 0.5e-13, max_iter = int(1e4),
                                 W1_init = W1_2,
                                 W2_init = W2_2)
print(J_3**0.5)

39570.033623801166


In [26]:
y_pred = simple_NN_pred(X_test_n, W1_3, W2_3)
f"testing error is {root_mean_squared_error(y_pred, y_test)}"

'testing error is 43625.04589447899'

## Result from linear reg

In [125]:
lm_h = LinearRegression().fit(X_train,y_train)
# use a data frame for output
results = pd.DataFrame({'training_RMSE':[],
                        'testing_RMSE':[]})
results.loc['linear',:] = [root_mean_squared_error(lm_h.predict(X_train), y_train.to_numpy()),
                           root_mean_squared_error(lm_h.predict(X_test), y_test.to_numpy())]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308


## Adjusting learning rate
We know that the learning rate affects the training significantly, so we will go through some common optimizaiton techniques for having better (adaptive) learning rates.

In [11]:
def simple_NN_l1(X, y, seed = 2, hidden_size = 5, max_iter = 30, alpha = 0.5, lr = 1e-13, **kwargs):
    '''
    X, y should be numpy arrays
    W1_init: optional argument for initialized weight
    W2_init: optional argument for initialized weight
    '''
    n, p = X.shape
    # reshape y to a column
    y = y.reshape([-1, 1])

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # random initialization if initial weights not provided
    np.random.seed(seed)
    w_scale = np.sqrt(6/(p+1))
    W1_rand = np.random.rand(p, p1) * w_scale 
    W2_rand = np.random.rand(p1, 1) * w_scale

    W1 = kwargs.get('W1_init', W1_rand)
    W2 = kwargs.get('W2_init', W2_rand)

    v1 = 0
    v2 = 0
    for i in range(max_iter):
        # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
        u1, h, u2 = forward_prop(X, W1, W2) # u2 is supposed to be approx y
    
        # compute cost
        J = J_MSE(u2, y)
    
        # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
        dW1, dW2 = backward_prop(X, y, u1, h, u2, W1, W2)
        
        # gradient descent with momentum
        v1 = alpha*v1 - lr*dW1
        v2 = alpha*v2 - lr*dW2
        W1 = W1 + v1
        W2 = W2 + v2

        # if alpha = 0, then this is the same as simple_NN()
        
    
    return W1, W2, J, u2

In [133]:
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n, y_train_n, hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, max_iter = int(4e4),                                 
                                 )
f"training error for using momentum is {J_m**0.5:.2f}"

'training error for using momentum is 30974.31'

In [134]:
# test error?
y_pred = simple_NN_pred(X_test_n, W1_m, W2_m)
f"testing error for using momentum is {root_mean_squared_error(y_pred, y_test):.2f}"

'testing error for using momentum is 34858.53'

In [135]:
results.loc['simple_NN_momentum',:] = [J_m**0.5,
                                       root_mean_squared_error(y_pred, y_test)]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308
simple_NN_momentum,30974.312249,34858.528183


## Stochastic Gradient Descent (SGD)
At each iteration of the gradient descent, the gradient will be **approximated** from only a minibatch of training data.

In [53]:
X_train.shape

(2197, 31)

We will let the batchsize be 200. The training data will be split into $2197/200\approx 11$ batches.

In [55]:
bs = 200 # batch size

In [58]:
# update parameters using first batch
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[:bs,:], y_train_n[:bs], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-12, max_iter = 100,                                 
                                 )
print(J_m**0.5)

44733.873546237104


In [60]:
# update parameters using 2nd batch
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[bs:(2*bs),:], y_train_n[bs:(2*bs)], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-12, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

36513.449038753664


In [61]:
# update parameters using 3rd batch
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[(2*bs):(3*bs),:], y_train_n[(2*bs):(3*bs)], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-12, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

43483.79189896798


In [62]:
# update parameters using 4th batch
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[(3*bs):(4*bs),:], y_train_n[(3*bs):(4*bs)], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-12, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

34400.57632125395


In [63]:
# update parameters using 5th batch
W1_m, W2_m, J_m, u2_m = simple_NN_l1(X_train_n[(4*bs):(5*bs),:], y_train_n[(4*bs):(5*bs)], hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-12, max_iter = 100,                                 
                                    W1_init = W1_m,
                                    W2_init = W2_m)
print(J_m**0.5)

36257.358443042685


...

We can keep going until we go through all 11 batches. In practice, we will make 2 modifications:

- Each batch is drawn randomly from the whole traning set. This is usually done by randomly shuffling the whole training set first.
- Only **one** iteration is run for a minibatch.

We will write a function with these two modifications in mind.

In [71]:
def simple_NN_SGD(X, y, seed = 2, hidden_size = 5, bs = 200, n_epoch = 30, alpha = 0.5, lr = 1e-13, **kwargs):
    '''
    X, y should be numpy arrays
    W1_init: optional argument for initialized weight
    W2_init: optional argument for initialized weight
    '''
    n, p = X.shape
    # reshape y to a column
    y = y.reshape([-1, 1])

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # random initialization if initial weights not provided
    np.random.seed(seed)
    w_scale = np.sqrt(6/(p+1))
    W1_rand = np.random.rand(p, p1) * w_scale 
    W2_rand = np.random.rand(p1, 1) * w_scale

    W1 = kwargs.get('W1_init', W1_rand)
    W2 = kwargs.get('W2_init', W2_rand)

    # initial velocity is often set to 0
    v1 = 0
    v2 = 0

    # number of batches
    n_bt = int(np.ceil(n/bs))
    
    for ei in range(n_epoch):
        # ei for epoch index
        # for each epoch
        # first shuffle X
        shuffled_idx = np.random.permutation(n)

        # the following for loop runs through the whole training set ONCE
        for bi in range(n_bt):
            # for the last batch, (bi+1)*bs most likely exceeds n. this is ok as numpy will automatically take the min of {n, (bi+1)*bs}
            bt_idx = shuffled_idx[(bi*bs):(bi+1)*bs] 
            X_bt = X[bt_idx,:]
            y_bt = y[bt_idx]
            # forward propargation
            u1, h, u2 = forward_prop(X_bt, W1, W2)
            # compute cost
            J = J_MSE(u2, y_bt)
            # backward propagation for computing gradient
            dW1, dW2 = backward_prop(X_bt, y_bt, u1, h, u2, W1, W2)
            # gradient descent with momentum
            v1 = alpha*v1 - lr*dW1
            v2 = alpha*v2 - lr*dW2
            W1 = W1 + v1
            W2 = W2 + v2
        ei  += 1                        
    
    return W1, W2, J, u2

In [122]:
W1_m, W2_m, J_m, u2_m = simple_NN_SGD(X_train_n, y_train_n, hidden_size = p1, seed = 3, alpha = 0.95, lr = 1e-13, n_epoch = 50000,                                 
                                     )
print(J_m**0.5)

31320.9393951049


In [123]:
y_pred = simple_NN_pred(X_test_n, W1_m, W2_m)
f"testing error is {root_mean_squared_error(y_pred, y_test):.2f}"

'testing error is 36371.81'